In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\python\data\raw data\yellow_tripdata_2015-01.csv")
print(df.shape)

(12748986, 19)


In [ ]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

df["trip_duration"] = (
    df["tpep_dropoff_datetime"] -
    df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["average_speed_mph"] = np.where(
    df["trip_duration"] > 0,
    df["trip_distance"] / (df["trip_duration"] / 60),
    np.nan
)

In [ ]:
cleaning_log= []

In [ ]:
def log_cleaning(rule_id, issue, before_rows, after_rows):
    cleaning_log.append({
        "Rule_ID": rule_id,
        "Issue": issue,
        "Rows_Before": before_rows,
        "Rows_After": after_rows,
        "Rows_Removed": before_rows - after_rows
    })

In [ ]:
# DQ-05 REMOVE DUPLICATES
rows_before = len(df)
df= df.drop_duplicates().copy()
rows_after = len(df)
log_cleaning(
    rule_id= "DQ-05",
    issue = "Remove Duplicate Records",
    before_rows= rows_before,
    after_rows= rows_after
)
print("Rows before :" , rows_before)
print("Rows after :" , rows_after)
print("Rows removed :" , rows_before-rows_after)


In [ ]:
# DQ-1 Remove corrupted distances
rows_before = len (df)
df = df[df["trip_distance"]<100].copy()
rows_after = len(df)
log_cleaning(
    rule_id= "DQ-05",
    issue= "Remove corrupted distances",
    before_rows= rows_before,
    after_rows= rows_after
)
print(f"Rows Before : {rows_before:,}")
print(f"Rows after : {rows_after:,}")
print(f"Rows removed : {rows_before-rows_after:,}")
print("Remainig Trips >100 miles ",(df["trip_distance"]>100).sum())


In [ ]:
# DQ-02 Remove trips greater than 10 hours
MAX_VALID_DURATION = 600
rows_before = len(df)
df = df[df["trip_duration"]<600].copy()
rows_after = len(df)
log_cleaning(
    rule_id= "DQ-02",
    issue= "Remove invalid distances",
    before_rows= rows_before,
    after_rows= rows_after
)
print(f"Rows Before : {rows_before:,}")
print(f"Rows after : {rows_after:,}")
print(f"Rows removed : {rows_before-rows_after:,}")
print(f"Remainig Trips > 10 HOURS :{(df["trip_distance"]>100).sum():,}")


In [ ]:
# DQ-07 : Remove Negative Trip Durations

rows_before = len(df)
df = df[df["trip_duration"] >= 0].copy()
rows_after = len(df)

log_cleaning(
    rule_id="DQ-07",
    issue="Negative Trip Duration",
    before_rows=rows_before,
    after_rows=rows_after
)
print(f"Rows Before : {rows_before:,}")
print(f"Rows After  : {rows_after:,}")
print(f"Rows Removed: {rows_before - rows_after:,}")
print("\nRemaining Negative Durations:",
      (df["trip_duration"] < 0).sum())

In [ ]:
#DQ-06 Impute missing surcharge values 

rows_before = len(df)
missing_before = df["improvement_surcharge"].isna().sum()
mode_value = df["improvement_surcharge"].mode()[0]
df['improvement_surcharge'] = df["improvement_surcharge"].fillna(mode_value)
missing_after= df["improvement_surcharge"].isna().sum()

log_cleaning(
    rule_id = "DQ-06",
    issue= "Impute Missing Improvement Surcharge",
    before_rows= rows_before,
    after_rows= rows_before
)
print(f"Missing Before : {missing_before:,}")
print(f"Missing After  : {missing_after:,}")
print(f"Mode value used: {mode_value:,}")

In [ ]:
# DQ-03 Flag the negative Fares
df['is_negative_fare'] = df['fare_amount']<0
negative_fare_Count = df['is_negative_fare'].sum()
log_cleaning(
    rule_id= "DQ-03",
    issue = "Flag the Negative Fare Records",
    before_rows=len(df),
    after_rows= len(df)
)
print(f"Negative Fare records flagged :{negative_fare_Count:,}")

In [ ]:
# Final Data Quality Validation
validation_report = pd.DataFrame({
    "Check":[
        "Duplicate Rows",
        "Trips > 100 Miles",
        "Trips > 10 Hours",
        "Negative Trip Duration",
        "Missing Improvement Surcharge",
        "Negative Fare Records (Flagged)"
    ],
    "Count":[
        df.duplicated().sum(),
        (df["trip_distance"] > 100).sum(),
        (df["trip_duration"] > 600).sum(),
        (df["trip_duration"] < 0).sum(),
        df["improvement_surcharge"].isna().sum(),
        df["is_negative_fare"].sum()
    ]
})
print(validation_report)
print(f"FINAL DATASET SHAPE :{df.shape}")

In [ ]:
df.to_csv("yellow_tripdata_2015_01_clean.csv" , index = False)
print("Clean dataset exported successfully.")

In [ ]:
cleaning_log_df = pd.DataFrame(cleaning_log)

cleaning_log_df.to_csv("cleaning_log.csv", index=False)

print("Cleaning log exported successfully.")


In [ ]:
df.rename(columns={"trip_duration" : "trip_duration_minutes"} , inplace= True )
df.drop(columns="average_speed_mph", inplace=True)
print(df.columns.tolist())
print(f"\nThe final shape of dataset :{df.shape}")

In [3]:
rows_before = len(df)
df = df[ 
    (df["fare_amount"] <= 1000) &
    (df["tip_amount"] <= 500) &
    (df["total_amount"] <= 1500) &
    (df["tolls_amount"] <= 200)
]
rows_removed = rows_before - len(df)
print(f"Rows removed :{rows_removed:,}")

Rows removed :34


In [ ]:
print(df[[
    "fare_amount",
    "tip_amount" ,
    "total_amount" ,
    "tolls_amount"
]].describe(percentiles=[0.99, 0.999]))

In [7]:
df.to_csv(
    r"C:\Users\HP\OneDrive\Desktop\python\notebooks\yellow_tripdata_2015_01_featured.csv",
    index=False
)